# Analyse network results for sector-coupled PyPSA-Earth
This notebook processes the latest optimization data specified in the config.yaml file and the network .nc file. It generates comprehensive plots and summaries to visualize the results.

Sources: 
- Statistics module: https://pypsa.readthedocs.io/en/latest/examples/statistics.html
- Templates and colors: https://github.com/pypsa-meets-earth/documentation/blob/main/notebooks/network_analysis.ipynb

File is needed:
* PyPSA network file (.nc) includes the heating, transprot, and electricity demand information.

## Import packages

In [ ]:
import yaml
import pypsa
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import os
import warnings

## Path settings
This section reads the config parameters from your config.yaml file and automatically reads the output of the optimization with those settings

In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)
# change current directory to parent folder
if not os.path.isdir("pypsa-earth"):
    os.chdir("../..")

PARENT = os.path.realpath("pypsa-earth/") + "/"
# Specify config name
CONFIG = "config-dz-sec-3H-NoCo2Sequ+NoBiomass"
config = yaml.safe_load(open(PARENT + "own-configs/" + CONFIG + ".yaml"))

In [ ]:
run_name = config["run"]["name"]
run_sector_name = config["run"]["sector_name"]                          
simpl = config["scenario"]["simpl"]    
clust = config["scenario"]["clusters"]   
ll = config["scenario"]["ll"]             
load_scale = config["load_options"]["scale"]               
opts = config["scenario"]["opts"]       
sopts = config["scenario"]["sopts"]              
planning = config["scenario"]["planning_horizons"]        
discountrate = config["costs"]["discountrate"]                
demand = config["scenario"]["demand"]                  
export_value = config["export"]["h2export"]                  

# Process each setting into a string representation 
simpl_str = "_".join(map(str, simpl))   
clust_str = "_".join(map(str, clust))    
ll_str = "l" + "_".join(map(str, ll))           
scale_str = f"lc{load_scale}"                           
opts_str = "_".join(map(str, opts))    
sopts_str = "_".join(map(str, sopts))                 
planning_str = "_".join(map(str, planning))               
dr_str = "_".join(map(str, discountrate))             
demand_str = "_".join(map(str, demand))                    
export_str = "_".join(map(str, export_value)) + "export"   

nc_file_name = (
    f"elec_s_{clust_str}_ec_{ll_str}_{opts_str}_{sopts_str}_"
    f"{planning_str}_{dr_str}_{demand_str}_{export_str}.nc"
)

### Analysis Network Setup

In [ ]:
# scenario_subpath = f"{run_name}/" if run_name else ""
scenario_subpath = f"{run_sector_name}/" if run_sector_name else ""
results_path = PARENT + f"results/{run_sector_name}/postnetworks/{nc_file_name}"
n = pypsa.Network(results_path)

## Data import check

List number of components by type

In [ ]:
for c in n.iterate_components(list(n.components.keys())[2:]):
    print("Component '{}' has {} entries".format(c.name,len(c.df)))

List the snapshots of the PyPSA network

In [ ]:
print(n.snapshots)
print(f"Time steps: " + str(len(n.snapshots)))

## Analyse energy system

### Transport Sector 
The demand of EVs, ICE (internal combustion engine) and FCEV (fuell cell)

In [ ]:
load_stats = n.statistics.energy_balance(comps=["Load"], aggregate_time="sum")
transport_stats_GWh = load_stats[load_stats.index.get_level_values('carrier').str.contains(
    'land transport oil(?! emissions)|land transport fuel cell|land transport EV', 
    case=True)]
transport_stats_GWh

In [ ]:
transport_stats_TWh = transport_stats_GWh/1000000

ax = transport_stats_TWh.plot.bar(
    title='Demand of EVs, ICE and FCEV',
    xlabel="",
    ylabel="Demand in TWh"
)

ax.set_xticklabels(transport_stats_GWh.index.get_level_values('carrier'), rotation=45, ha="right")
ax.grid(True) 
plt.show()

### Heating Sector 
The demand of various heating subsectors

In [ ]:
load_stats = n.statistics.energy_balance(comps=["Load"], aggregate_time="sum")
# Wrong naminh (GWh, should be MWh)
heating_stats_GWh = load_stats[load_stats.index.get_level_values('carrier').str.contains('heat', case=False)]
heating_stats_GWh

In [ ]:
# Filter links whose carrier contains 'co2'
co2_links = n.links[n.links.carrier.str.contains('co2', case=False, na=False)].index
# Sum the p0 values for these links over all snapshots
(n.links_t.p0[co2_links].sum().sum())*144 / 1e6  # Convert to Mt

In [ ]:
(load_stats[load_stats.abs() >= 1000000].sort_values(ascending=True) / 1e6).plot.bar()

In [ ]:
heating_stats_TWh = heating_stats_GWh/1000000

ax = heating_stats_TWh.plot.bar(
    title='The demand of various heating subsectors',
    xlabel="",
    ylabel="Demand in TWh"
)

ax.set_xticklabels(heating_stats_TWh.index.get_level_values('carrier'), rotation=45, ha="right")
ax.grid(True)
plt.show()

How is heating energy supplied (heat pump, district heating, resistive heater, gas boiler)

In [ ]:
n.statistics.supply() / 1e6

In [ ]:
(n.statistics.supply(comps=["Link"]).loc[
    lambda x: x.index.get_level_values('carrier').str.contains('heat|boiler|pump|chp', case=False)
] / 1e3).sort_values(ascending=False).plot.bar()
plt.yscale('log')
plt.show()

(n.statistics.supply(comps=["Link"]).loc[
    lambda x: x.index.get_level_values('carrier').str.contains('heat|boiler|pump|chp', case=False)
] / 1e6).sum()

In [ ]:
biomass_gens = n.generators[n.generators.carrier.str.contains("biomass", case=False)]
# Get the total output for each generator (sum over all snapshots)
biomass_output = n.generators_t.p[biomass_gens.index].sum()  # MWh
# Remove all outputs < 1e6 MWh
biomass_output = biomass_output[biomass_output >= 1e6]
biomass_output_sorted = biomass_output.sort_values(ascending=False)

plt.figure(figsize=(12, 5))
biomass_output_sorted.plot.bar()
plt.ylabel("Total Output [MWh]")
plt.title("Total Output of Biomass Generators (Descending)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
heating_supply_stats = n.statistics.supply(comps=["Link"], aggregate_time="sum").loc[
    lambda x: x.index.get_level_values('carrier').str.contains('heat|boiler|pump|chp', case=False)
]

# Only the Gas boiler, heat pump, CHP, and resistive heater technologies are considered here
heating_types = ['Gas Boiler', 'Heat Pump', 'Resistive Heater', 'CHP']
heating_sums = [heating_supply_stats[heating_supply_stats.index.get_level_values('carrier').str.contains(heatingtype, case=False)].sum() for heatingtype in heating_types]

# ALternative: Save as dict

heating_sums_dict = {
    heating_type: heating_supply_stats.loc[
        heating_supply_stats.index.get_level_values('carrier').str.contains(heating_type, case=False)
    ].sum()
    for heating_type in heating_types
}

for ht, sum_value in zip(heating_types, heating_sums):
    print(f"{ht}: {sum_value/1e6} TWh")

plt.figure(figsize=(6, 6))
plt.pie(heating_sums, autopct='%1.1f%%')
plt.title('Heating Supply Distribution')
plt.legend(heating_types, loc='upper left', bbox_to_anchor=(1, 0.8))
plt.axis('equal')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(heating_sums_dict.keys(), [v / 1e6 for v in heating_sums_dict.values()])
plt.ylabel('Supply in TWh')
plt.title('Heating Supply by Technology')
plt.grid(axis='y', linestyle='dotted')
plt.tight_layout()
plt.show()

In [ ]:
# To get the energy balance for 'load', filter after calling energy_balance()
energy_balance = n.statistics.energy_balance()
# Exclude entries where bus_carrier is 'co2' and use absolute values
filtered = energy_balance[
    (energy_balance.index.get_level_values('component') == 'Load') &
    (energy_balance.index.get_level_values('bus_carrier') != 'co2')
].abs().sort_values(ascending=False) / 1e6
filtered

In [ ]:
chp_links = n.links[n.links.carrier.str.contains('chp', case=False, na=False)]
print(chp_links)

In [ ]:
# Show all carrier entries with 'heat' in them using n.links_t

heat_link_mask = n.links.carrier.str.contains('heat', case=False, na=False)
heat_links = n.links[heat_link_mask].index

# Show demand (input power, p0) for those links, summed over all snapshots
heat_links_demand = ((n.links_t.p0[heat_links])*144).sum()  # MWh
heat_links_demand.sum() / 1e6



### Electricity Sector 
The share of different carriers (capacity and energy)

In [ ]:
def color_matching(stats):
    colors = {key.lower(): value.lower() for key, value in config["plotting"]["tech_colors"].items()}
    nice_names = {value.lower(): key for key, value in config["plotting"]["nice_names"].items()}
    color_list = []
    for carrier in stats.index.get_level_values("carrier"):
        original_name = carrier.lower()
        key_name = nice_names.get(original_name, original_name)
        color = colors.get(key_name.lower(), 'gray')
        color_list.append(color)
    return color_list

In [ ]:
capacity_stats = n.statistics.installed_capacity(comps=["Generator"])
# Remove carrier with the capacity = 0
capacity_stats_non_zero = capacity_stats[capacity_stats != 0].dropna()
capacity_stats_plus = n.statistics.installed_capacity(comps=["StorageUnit"])
combined_capacity_stats = pd.concat([capacity_stats_non_zero, capacity_stats_plus])
# Remove loads from the capacity stats
combined_capacity_stats = combined_capacity_stats.drop(('Generator', 'load'), errors='ignore')
combined_capacity_stats

In [ ]:
color_list = color_matching(combined_capacity_stats)

combined_capacity_stats.plot.pie(
    title="Share of different carriers by capacity",
    labels=None,
    autopct='%1.1f%%',
    pctdistance=0.85,
    startangle=90,
    ylabel="",
    colors=color_list,
)

custom_labels = combined_capacity_stats.index.get_level_values("carrier")
plt.legend(custom_labels, loc='upper left', bbox_to_anchor=(1, 0.8))


In [ ]:
energy_stats = n.statistics.energy_balance(comps=["Generator"])
electricity_energy_stats = energy_stats[energy_stats.index.get_level_values('bus_carrier') == 'AC']
electricity_energy_stats_plus = n.statistics.energy_balance(comps=["StorageUnit"])
combined_electricity_energy_stats = pd.concat([electricity_energy_stats, electricity_energy_stats_plus])
# Remove load from the electricity geneation stats
combined_electricity_energy_stats = combined_electricity_energy_stats.drop(('Generator', 'load', 'AC'), errors='ignore')
combined_electricity_energy_stats

In [ ]:
color_list = color_matching(combined_electricity_energy_stats)

combined_electricity_energy_stats.plot.pie(
    title='Share of Different Carriers by Energy',
    labels=None,
    autopct='%1.1f%%',
    pctdistance=0.85,
    startangle=90,
    ylabel="",
    colors=color_list,  
)

custom_labels = combined_electricity_energy_stats.index.get_level_values("carrier")
plt.legend(custom_labels, loc='upper left', bbox_to_anchor=(1, 0.8))

## Own investigation

In [ ]:
n.statistics()

# Plot all generator's supply
gen_supply = n.statistics.energy_balance(comps=["Generator"], aggregate_time="sum")
# Filter out zero or negative values if needed
gen_supply = gen_supply[gen_supply > 0]
# Filter out carriers < 10 MWh 
gen_supply = gen_supply[gen_supply > 10]
# Remove all 'load' from the index before plotting
gen_supply = gen_supply[gen_supply.index.get_level_values('carrier') != 'load']

# Sort by amount descending
gen_supply = gen_supply.sort_values(ascending=False)

# Plot
plt.rcParams.update({'font.size': 18})
fig, ax = plt.subplots(figsize=(10, 6))
ax = (gen_supply / 1e6).plot.bar(
    # title="Total Supply by Generation Type",
    ylabel="Energy Supplied in TWh",
    xlabel="Generation Type"
)
# Hide the top and right frame lines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
# Add dotted horizontal lines at all y ticks
plt.gca().yaxis.grid(True, linestyle='dotted', linewidth=0.5, color='gray')
#for y in plt.gca().get_yticks():
#    plt.axhline(y=y, color='gray', linestyle='dotted', linewidth=0.5)
# Limit the y-axis to 300 TWh
plt.ylim(0, 1500)
ax.set_xticklabels(gen_supply.index.get_level_values('carrier'), rotation=45, ha="right")
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Filter for biomass CHP units in chp_links
biomass_chp_mask = chp_links['carrier'].str.contains('biomass', case=False, na=False)
biomass_chp = chp_links[biomass_chp_mask]

# Get both Installed Capacity and Optimal Capacity
installed_capacity = biomass_chp['p_nom']
optimized_capacity = biomass_chp['p_nom_opt']

# Sort by optimized capacity descending
sorted_idx = optimized_capacity.sort_values(ascending=False).index

plt.figure(figsize=(14, 6))
plt.bar(biomass_chp.loc[sorted_idx].index, installed_capacity.loc[sorted_idx] / 1e3, width=0.4, label='Installed Capacity [GW]')
plt.bar(biomass_chp.loc[sorted_idx].index, optimized_capacity.loc[sorted_idx] / 1e3, width=0.4, label='Optimized Capacity [GW]', alpha=0.7)
plt.ylabel('Capacity [GW]')
plt.title('Installed vs Optimized Capacity of Biomass CHP Units')
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Get all generators' installed and optimized capacities, grouped by carrier
installed_by_carrier = n.generators.groupby('carrier')['p_nom'].sum()
optimized_by_carrier = n.generators.groupby('carrier')['p_nom_opt'].sum()

# Combine into a DataFrame for plotting (convert to GW)
capacity_df = pd.DataFrame({
    'Installed Capacity [GW]': installed_by_carrier / 1e3,
    'Optimized Capacity [GW]': optimized_by_carrier / 1e3
})

# Remove 'load' and carriers with zero capacity
capacity_df = capacity_df.drop('load', errors='ignore')
capacity_df = capacity_df[(capacity_df > 0).any(axis=1)]

# Sort descending by optimized capacity
capacity_df = capacity_df.sort_values('Optimized Capacity [GW]', ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(8, 15))

capacity_df.plot.bar(ax=ax)
plt.ylabel('Capacity [GW]')
plt.title('Installed (Brownfield) and Optimized Capacity by Generator Carrier')
plt.grid(axis='y', linestyle='dotted')
plt.tight_layout()
plt.show()


In [ ]:
biomass_chp_buses = biomass_chp['bus0'].unique()
links_connected = n.links[
    n.links['bus0'].isin(biomass_chp_buses) | n.links['bus1'].isin(biomass_chp_buses)
]
links_connected

In [ ]:
# Find all components (links, generators, stores, loads, etc.) connected to 'DZ.22_1_AC solid biomass bua'
bus_name = 'DZ.22_1_AC solid biomass'

connected = {}

# Links
links_connected = n.links[
    (n.links['bus0'] == bus_name) |
    (n.links['bus1'] == bus_name) |
    (n.links.get('bus2', pd.Series([None]*len(n.links))) == bus_name) |
    (n.links.get('bus3', pd.Series([None]*len(n.links))) == bus_name) |
    (n.links.get('bus4', pd.Series([None]*len(n.links))) == bus_name)
]
if not links_connected.empty:
    connected['links'] = links_connected

# Generators
generators_connected = n.generators[n.generators['bus'] == bus_name]
if not generators_connected.empty:
    connected['generators'] = generators_connected

# Stores
stores_connected = n.stores[n.stores['bus'] == bus_name]
if not stores_connected.empty:
    connected['stores'] = stores_connected

# Loads
loads_connected = n.loads[n.loads['bus'] == bus_name]
if not loads_connected.empty:
    connected['loads'] = loads_connected

# StorageUnits
if hasattr(n, 'storage_units'):
    storage_units_connected = n.storage_units[n.storage_units['bus'] == bus_name]
    if not storage_units_connected.empty:
        connected['storage_units'] = storage_units_connected

# Print results
for comp, df in connected.items():
    print(f"\n{comp.upper()} connected to {bus_name}:")
    display(df)

In [ ]:
# Filter for CO2 sources and sinks in the energy_stats
co2_stats = energy_stats[energy_stats.index.get_level_values('bus_carrier').str.contains('co2', case=False)]

# Separate sources (positive values) and sinks (negative values)
co2_sources = co2_stats[co2_stats > 0]
co2_sinks = co2_stats[co2_stats < 0]

# Combine for plotting
co2_plot = pd.concat([co2_sources, co2_sinks])

# Prepare labels for the plot
labels = [
    f"{idx[1]} ({'source' if val > 0 else 'sink'})"
    for idx, val in co2_plot.items()
]

plt.figure(figsize=(8, 5))
(co2_plot).plot.bar(
    color=['#b80404' if val > 0 else '#4adbc8' for val in co2_plot],
    ylabel="CO₂ in Mt (?)",
    xlabel="CO₂ Process",
    title="CO₂ Sources and Sinks"
)
plt.xticks(ticks=range(len(labels)), labels=labels, rotation=45, ha="right")
plt.tight_layout()
plt.grid(axis='y', linestyle='dotted')
plt.show()

In [ ]:
energy_stats

In [ ]:
stats = n.statistics()
# Extract co2 entries from the statistics
co2_entries = stats.loc[stats.index.to_frame().astype(str).apply(lambda row: row.str.contains('co2', case=False).any(), axis=1)]
#print(co2_entries)

co2_stored = co2_entries.loc[('Store', 'co2 stored')].Withdrawal

print("The total amount of CO2 stored in the system is: {:.2f} Mt".format(co2_stored / 1e6))

# To access the Capital Expenditure for ('Store', 'co2 stored'), use:
co2_entries.loc[('Store', 'co2 stored'), 'Capital Expenditure'] / 1e6

In [ ]:
# Plot the primary energy sources for the heating sector
# We'll use heating_supply_stats and heating_types from previous cells

plt.figure(figsize=(8, 6))
plt.bar(heating_types, [v / 1e6 for v in heating_sums], color=['#b80404', '#6895dd', '#74c6f2', '#235ebc'])
plt.ylabel('Supply in TWh')
plt.title('Primary Energy Sources for Heating Sector')
plt.grid(axis='y', linestyle='dotted')
plt.tight_layout()
plt.show()

In [ ]:
# Extract all CHP units from n.links_t
# The primary energy carrier is in n.links.carrier, and the volume is in n.links_t.p0 (input power)
chp_mask = n.links.carrier.str.contains('chp', case=False, na=False)
chp_links = n.links[chp_mask]

# For each CHP, get its name, primary energy carrier, and total input volume (sum over all snapshots)
chp_volumes = n.links_t.p0[chp_links.index].sum() * 144
chp_info = pd.DataFrame({
    "Primary Carrier": chp_links.bus0,
    "Volume (MWh)": chp_volumes
}).reset_index()

print(chp_info)

chp_volumes.sum() / 1e6